# 52. 제품별 통합 불량 예측 모델 — 학습 · 테스트 · 베이스라인 비교 · unlabeled 일반화

**현장은 불량 유형을 모름 → 제품마다 검사 목록 1개.** 안에서는 유형별 탐지기, 밖으로는 하나의 검사 순위

| 제품 | 통합 모델 (P) | 내부 탐지기 → 추정 원인 |
|---|---|---|
| CN7 | 최종 모델 경보를 맨 위 + 나머지 std4 흔들림 순 (라벨 0개) | 최종 모델 경보 → 미성형(재료부족)형 / std4 → 가스(흔들림)형 |
| RG3 | 가스 RF 순위 : N1 순위 = 68 : 32 로 번갈아 합친 순위 | 가스 RF → 가스형 / N1 → 미성형형 |

**비교 (같은 조건, '불량 전체'가 기준 지표)**: B0 무작위 · B1 최종 모델 단독 · B2 단일 RF '불량 전체' (LORO / 시간 순) · B3 단일 RF + 정상 보강 · P 통합
- 평가 틀: `2차`(공식 2차 샷, RG3 지도학습도 2차만) / `전체`(1차+2차), 검사 5·10·15%, 시드 3개
- 가스+미성형은 가스·미성형 양쪽에 셈 (불량 전체 수는 그대로)
- 실행 시간: 2개 틀 × 2개 제품 × 시드 3개 → 수 분~10분

In [1]:
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 300)
%load_ext autoreload
%autoreload 2
import unified_model as UM
R = UM.run(frames=('2차', '전체'), seeds=(0, 1, 2))

[2차] cn7 완료
[2차] rg3 완료
[전체] cn7 완료
[전체] rg3 완료


## ① 2장 비교표 — 제품 × 모델, 검사 10%, 불량 전체 기준 (2차 / 전체 나란히)

In [2]:
display(UM.headline(R).round(3))

,제품,모델,불량 [2차],Precision [2차],Recall [2차],F1 [2차],무작위 대비(배) [2차],가스 [2차],미성형 [2차],불량 [전체],Precision [전체],Recall [전체],F1 [전체],무작위 대비(배) [전체],가스 [전체],미성형 [전체],ROC-AUC [전체],R-Precision [전체]
0,cn7,B0 무작위 검사,1.6/14,0.026,0.114,0.043,1.000,1.4/13,0.6/3,1.7/15,0.009,0.113,0.016,1.000,1.4/13,0.7/4,NaN,NaN
1,cn7,B1 최종 모델 점수 (비지도),5.0/14,0.082,0.357,0.133,3.125,4.0/13,3.0/3,5.0/15,0.025,0.333,0.047,2.941,4.0/13,3.0/4,0.569,0.200
2,cn7,B2 단일 RF 불량전체 (LORO),0.0/14,0.000,0.000,0.000,0.000,0.0/13,0.0/3,0.7/15,0.003,0.044,0.006,0.392,0.0/13,0.7/4,0.091,0.000
3,cn7,B2 단일 RF 불량전체 (시간 순),0.0/14,0.000,0.000,0.000,0.000,0.0/13,0.0/3,0.0/15,0.000,0.000,0.000,0.000,0.0/13,0.0/4,0.085,0.000
4,cn7,B3 단일 RF 불량전체 + 정상 보강 (LORO),0.0/14,0.000,0.000,0.000,0.000,0.0/13,0.0/3,1.0/15,0.005,0.067,0.009,0.588,0.0/13,1.0/4,0.091,0.000
5,cn7,P 통합: 경보 먼저 + std4,6.0/14,0.098,0.429,0.160,3.750,5.0/13,3.0/3,8.0/15,0.040,0.533,0.075,4.706,7.0/13,3.0/4,0.736,0.200
6,rg3,(참고) 가스 RF 단독 (LORO),9.3/25,0.156,0.373,0.220,3.889,7.7/17,1.7/8,9.3/25,0.156,0.373,0.220,3.889,7.7/17,1.7/8,0.629,0.160
7,rg3,B0 무작위 검사,2.4/25,0.040,0.096,0.056,1.000,1.6/17,0.8/8,2.4/25,0.040,0.096,0.056,1.000,1.6/17,0.8/8,NaN,NaN
8,rg3,B1 최종 모델 점수 (비지도),2.0/25,0.033,0.080,0.047,0.833,1.0/17,1.0/8,2.0/25,0.033,0.080,0.047,0.833,1.0/17,1.0/8,0.554,0.040
9,rg3,B2 단일 RF 불량전체 (LORO),6.0/25,0.100,0.240,0.141,2.500,5.3/17,0.7/8,6.0/25,0.100,0.240,0.141,2.500,5.3/17,0.7/8,0.659,0.080


## ② 검사 비율별 (5·10·15%) — 샷 단위

In [3]:
for fr in ('2차', '전체'):
    print(f'===== {fr}')
    display(R['curves'][R['curves']['평가틀'] == fr].round(3))

===== 2차


,평가틀,제품,모델,검사 비율,검사 샷,불량,Precision,Recall,F1,무작위 대비(배),가스,미성형
0,2차,cn7,B0 무작위 검사,5%,31,1.0/14,0.034,0.075,0.047,1.000,0.8/13,0.4/3
1,2차,cn7,B0 무작위 검사,10%,61,1.6/14,0.026,0.114,0.043,1.000,1.4/13,0.6/3
2,2차,cn7,B0 무작위 검사,15%,91,2.3/14,0.025,0.164,0.044,1.000,2.0/13,0.8/3
3,2차,cn7,B1 최종 모델 점수 (비지도),5%,31,3.0/14,0.097,0.214,0.133,2.857,2.0/13,3.0/3
4,2차,cn7,B1 최종 모델 점수 (비지도),10%,61,5.0/14,0.082,0.357,0.133,3.125,4.0/13,3.0/3
5,2차,cn7,B1 최종 모델 점수 (비지도),15%,91,5.0/14,0.055,0.357,0.095,2.174,4.0/13,3.0/3
6,2차,cn7,B2 단일 RF 불량전체 (LORO),5%,31,0.0/14,0.000,0.000,0.000,0.000,0.0/13,0.0/3
7,2차,cn7,B2 단일 RF 불량전체 (LORO),10%,61,0.0/14,0.000,0.000,0.000,0.000,0.0/13,0.0/3
8,2차,cn7,B2 단일 RF 불량전체 (LORO),15%,91,0.0/14,0.000,0.000,0.000,0.000,0.0/13,0.0/3
9,2차,cn7,B2 단일 RF 불량전체 (시간 순),5%,31,0.0/14,0.000,0.000,0.000,0.000,0.0/13,0.0/3


===== 전체


,평가틀,제품,모델,검사 비율,검사 샷,불량,Precision,Recall,F1,무작위 대비(배),가스,미성형
42,전체,cn7,B0 무작위 검사,5%,99,1.1/15,0.011,0.073,0.019,1.000,0.8/13,0.5/4
43,전체,cn7,B0 무작위 검사,10%,198,1.7/15,0.009,0.113,0.016,1.000,1.4/13,0.7/4
44,전체,cn7,B0 무작위 검사,15%,297,2.4/15,0.008,0.157,0.015,1.000,1.9/13,0.9/4
45,전체,cn7,B1 최종 모델 점수 (비지도),5%,99,4.0/15,0.040,0.267,0.070,3.636,3.0/13,3.0/4
46,전체,cn7,B1 최종 모델 점수 (비지도),10%,198,5.0/15,0.025,0.333,0.047,2.941,4.0/13,3.0/4
47,전체,cn7,B1 최종 모델 점수 (비지도),15%,297,5.0/15,0.017,0.333,0.032,2.128,4.0/13,3.0/4
48,전체,cn7,B2 단일 RF 불량전체 (LORO),5%,99,0.0/15,0.000,0.000,0.000,0.000,0.0/13,0.0/4
49,전체,cn7,B2 단일 RF 불량전체 (LORO),10%,198,0.7/15,0.003,0.044,0.006,0.392,0.0/13,0.7/4
50,전체,cn7,B2 단일 RF 불량전체 (LORO),15%,297,1.0/15,0.003,0.067,0.006,0.426,0.0/13,1.0/4
51,전체,cn7,B2 단일 RF 불량전체 (시간 순),5%,99,0.0/15,0.000,0.000,0.000,0.000,0.0/13,0.0/4


## ③ 공식 2차 행(LH/RH) 단위 — 대회 평가 형태

In [4]:
display(R['rows'].round(3))

,평가틀,제품,모델,검사 비율,검사 샷,불량,Precision,Recall,F1,무작위 대비(배),가스,미성형
0,2차,cn7,B0 무작위 검사,5%,61,1.1/17,0.018,0.065,0.028,1.000,0.8/13,0.2/4
1,2차,cn7,B0 무작위 검사,10%,122,1.7/17,0.014,0.100,0.024,1.000,1.3/13,0.4/4
2,2차,cn7,B0 무작위 검사,15%,182,2.7/17,0.015,0.159,0.027,1.000,2.2/13,0.4/4
3,2차,cn7,B1 최종 모델 점수 (비지도),5%,61,6.0/17,0.098,0.353,0.154,5.455,2.0/13,4.0/4
4,2차,cn7,B1 최종 모델 점수 (비지도),10%,122,8.0/17,0.066,0.471,0.115,4.706,4.0/13,4.0/4
5,2차,cn7,B1 최종 모델 점수 (비지도),15%,182,8.0/17,0.044,0.471,0.080,2.963,4.0/13,4.0/4
6,2차,cn7,B2 단일 RF 불량전체 (LORO),5%,61,0.0/17,0.000,0.000,0.000,0.000,0.0/13,0.0/4
7,2차,cn7,B2 단일 RF 불량전체 (LORO),10%,122,0.0/17,0.000,0.000,0.000,0.000,0.0/13,0.0/4
8,2차,cn7,B2 단일 RF 불량전체 (LORO),15%,182,0.0/17,0.000,0.000,0.000,0.000,0.0/13,0.0/4
9,2차,cn7,B2 단일 RF 불량전체 (시간 순),5%,61,0.0/17,0.000,0.000,0.000,0.000,0.0/13,0.0/4


## ④ 검사 비율 무관 지표 (R-Precision · ROC-AUC · AP)

In [5]:
display(R['rank'].round(3))

,평가틀,제품,모델,seed,불량 수,R-Precision,ROC-AUC,AP,무작위 기준(불량률),Precision@Recall25%,Recall25% 필요 검사 비율,Precision@Recall50%,Recall50% 필요 검사 비율,Precision@Recall75%,Recall75% 필요 검사 비율
0,2차,cn7,B1 최종 모델 점수 (비지도),0.0,14.0,0.214,0.609,0.247,0.023,0.103,0.064,0.027,0.436,0.028,0.637
1,2차,cn7,B2 단일 RF 불량전체 (LORO),1.0,14.0,0.000,0.105,0.013,0.023,0.008,0.830,0.013,0.889,0.019,0.954
2,2차,cn7,B2 단일 RF 불량전체 (시간 순),1.0,14.0,0.000,0.105,0.013,0.023,0.008,0.830,0.013,0.889,0.019,0.954
3,2차,cn7,B3 단일 RF 불량전체 + 정상 보강 (LORO),1.0,14.0,0.000,0.105,0.013,0.023,0.008,0.830,0.013,0.889,0.019,0.954
4,2차,cn7,P 통합: 경보 먼저 + std4,1.0,14.0,0.214,0.708,0.265,0.023,0.105,0.063,0.106,0.109,0.030,0.614
5,2차,rg3,B1 최종 모델 점수 (비지도),0.0,25.0,0.040,0.554,0.048,0.042,0.041,0.286,0.050,0.440,0.056,0.574
6,2차,rg3,B2 단일 RF 불량전체 (LORO),1.0,25.0,0.080,0.659,0.092,0.042,0.097,0.129,0.087,0.252,0.065,0.496
7,2차,rg3,B2 단일 RF 불량전체 (시간 순),1.0,25.0,0.093,0.618,0.087,0.042,0.105,0.114,0.053,0.412,0.053,0.609
8,2차,rg3,B3 단일 RF 불량전체 + 정상 보강 (LORO),1.0,25.0,0.147,0.632,0.092,0.042,0.072,0.165,0.062,0.356,0.061,0.530
9,2차,rg3,(참고) 가스 RF 단독 (LORO),1.0,25.0,0.160,0.629,0.119,0.042,0.203,0.060,0.058,0.383,0.053,0.606


## ⑤ 추정 원인 — 검사 10% 안에서 '어느 탐지기가 올렸나 × 실제 유형' (작업자 점검 방향 안내용)

In [6]:
display(R['source'])

실제,탐지기(추정 원인),가스,가스+미성형,미성형,정상,검사 샷,제품,평가틀,모델
0,std4 흔들림,3,0.0,0,53,56,cn7,2차,P 통합: 경보 먼저 + std4
1,최종 모델 경보,0,2.0,1,2,5,cn7,2차,P 통합: 경보 먼저 + std4
2,N1,1,NaN,1,17,19,rg3,2차,P 통합: 가스 RF(LORO) + N1
3,가스 RF,6,NaN,2,33,41,rg3,2차,P 통합: 가스 RF(LORO) + N1
4,N1,1,NaN,1,16,18,rg3,2차,P 통합: 가스 RF(시간 순) + N1
5,가스 RF,5,NaN,3,34,42,rg3,2차,P 통합: 가스 RF(시간 순) + N1
6,std4 흔들림,5,0.0,0,185,190,cn7,전체,P 통합: 경보 먼저 + std4
7,최종 모델 경보,0,2.0,1,5,8,cn7,전체,P 통합: 경보 먼저 + std4
8,N1,1,NaN,1,17,19,rg3,전체,P 통합: 가스 RF(LORO) + N1
9,가스 RF,6,NaN,2,33,41,rg3,전체,P 통합: 가스 RF(LORO) + N1


## ⑥ 운영 루프 (100샷 묶음마다 10% 검사) — 통합 순위 고정 vs 불량 발견 시 앞뒤 5샷 확장

In [7]:
display(R['operation'].round(3))

,제품,기본 점수,전략,seed,검사 샷,찾은 불량,전체 불량,Precision,Recall,F1,가스,가스 전체,미성형,미성형 전체,평가틀,모델
0,cn7,통합 모델,S1 고정 순위,0,61,5,14,0.082,0.357,0.133,4,13,3,3,2차,P 통합: 경보 먼저 + std4
1,cn7,통합 모델,S2 고정 + 주변 확장,0,61,8,14,0.131,0.571,0.213,7,13,3,3,2차,P 통합: 경보 먼저 + std4
2,rg3,통합 모델,S1 고정 순위,0,62,9,25,0.145,0.360,0.207,5,17,4,8,2차,P 통합: 가스 RF(시간 순) + N1
3,rg3,통합 모델,S2 고정 + 주변 확장,0,62,7,25,0.113,0.280,0.161,4,17,3,8,2차,P 통합: 가스 RF(시간 순) + N1
4,cn7,통합 모델,S1 고정 순위,0,199,5,15,0.025,0.333,0.047,4,13,3,4,전체,P 통합: 경보 먼저 + std4
5,cn7,통합 모델,S2 고정 + 주변 확장,0,199,8,15,0.040,0.533,0.075,7,13,3,4,전체,P 통합: 경보 먼저 + std4
6,rg3,통합 모델,S1 고정 순위,0,62,9,25,0.145,0.360,0.207,5,17,4,8,전체,P 통합: 가스 RF(시간 순) + N1
7,rg3,통합 모델,S2 고정 + 주변 확장,0,62,7,25,0.113,0.280,0.161,4,17,3,8,전체,P 통합: 가스 RF(시간 순) + N1


## ⑦ 신뢰성 — 통합 모델 불량 포착이 우연보다 많은가
- CN7: 라벨을 안 쓰므로 선택 고정 + 라벨만 Run 안 원형 이동 (2,000번)
- RG3: 가스 RF·N1까지 재학습하는 순열 (20번에 10~20분, 선택)

In [8]:
display(R['perm_cn7'][['평가틀', '제품', '모델', '검사 비율', '불량', 'p_불량', '무작위 평균_불량']].round(3))

,평가틀,제품,모델,검사 비율,불량,p_불량,무작위 평균_불량
0,2차,cn7,P 통합: 경보 먼저 + std4,10%,6/14,0.216,3.11
1,전체,cn7,P 통합: 경보 먼저 + std4,10%,8/15,0.152,3.59


In [11]:
print(UM.perm_rg3(frame='2차', n=20, forward=True))

  순열 0
  순열 5
  순열 10
  순열 15
{'관측 불량(검사 10%)': 10, '무작위 평균': 2.4, '무작위 95%': 4.100000000000001, 'p': 0.047619047619047616}


## ⑧ unlabeled 일반화 — 통합 목록을 unlabeled 전체 기간에 적용 (labeled 날짜 제외가 독립 검증)

In [10]:
import semi_final as SF
P = SF.fit_deploy(seed=0)
U = UM.unlabeled_unified(P, products=('cn7', 'rg3'))
display(U['corr'].round(3))

[cn7] 배포용 학습 저장 → ./outputs/semi_final/deploy_cn7.pkl
[rg3] 배포용 학습 저장 → ./outputs/semi_final/deploy_rg3.pkl
[cn7] unlabeled 통합 목록 완료
[rg3] unlabeled 통합 목록 완료


,제품,날짜 범위,날짜 수,"ρ(경보율, 불량률)",p,"ρ(경보 수, 불량 수)",p(수)
0,cn7,labeled 날짜 제외,12,-0.032,0.926,0.060,0.856
1,cn7,전체 날짜,15,0.072,0.804,0.221,0.426
2,rg3,labeled 날짜 제외,18,0.879,0.000,0.853,0.000
3,rg3,전체 날짜,21,0.818,0.000,0.733,0.000


## 읽는 법
1. **①이 보고서 2장 핵심 표**: P(통합)가 B1·B2·B3보다 '불량 전체'를 더 잡아야 '유형별 탐지기 → 통합'이 정당화됨
   - B2(단일 RF 불량 전체)가 CN7에서 거의 못 잡는 건 구조적: 불량 15개 중 14개가 R1 → R1을 빼면 학습할 불량이 없음
2. **무작위 대비(배)**로 검사 효율을 한 줄로 설명 (예: 같은 10% 검사로 무작위의 4배)
3. **⑤ 추정 원인**: '최종 모델 경보'로 올라온 샷이 실제로 미성형인지, '가스 RF'가 가스인지 → 4장 '작업자 지원'(어디를 점검할지) 근거
4. **⑥**: CN7은 확장으로 추가 포착, RG3는 확장 없이 운영 (49b)
5. **⑧**: RG3 ρ가 양수·유의하면 '처음 보는 날에도 불량 많은 날을 가려낸다' (일반화 근거), CN7은 일반화 미확인을 한계로